In [31]:
import os
from pathlib import Path

from mpi4py import MPI
from petsc4py import PETSc

import numpy as np
import gmsh
import math
import dolfinx
import ufl

from ufl import *
from dolfinx import *
from basix.ufl import element, mixed_element
from dolfinx.io import XDMFFile

from dolfinx import default_real_type, log, plot
from dolfinx.fem.petsc import NonlinearProblem, assemble_matrix
from dolfinx.io import XDMFFile
from dolfinx.io import gmsh as gmshio
from dolfinx.mesh import CellType, create_unit_square
from dolfinx import fem
from ufl import TrialFunction, TestFunction


from mpi4py import MPI

In [ ]:
# brick&mesh parameters
length = 3
width = 1
height = 1
fine_grid = 0.005             # mesh size for the hit region
coarse_grid = 60 * fine_grid # mesh size for the brick parts that are less interesting in the cutting process



# simulation parameters
dt = 2 * 10**(-6) # time step
time_strike = 0 # time at which the strike occurs
strike_duration = 0.5*10**(-3)
time_total = 1000 * dt
velocity_0 = 1  # [m/s], initial velocity applied onto the strike region


# physical values
E = 10 * 10**9 # [Pa] Young module of a brick
nu = 0.3 # [-] Poisson's ratio
Gc = 1 * 10**2 # [I/m^2] fracture toughness
sigma_c = 3 * 10**6 # [Pa] tensile strength of a brick (with l)
rho = 2000 # [kg/m^3] brick density


length0 = 27/256 * E * Gc * sigma_c**(-2)   # length scale
lambda_ = E*nu/((1+nu)*(1 - 2*nu))           # first lame constant
mu = E / (2 * (1+nu))                       # second lame consant
K = lambda_ + 2/3 * mu # bulk modulus


print(f"comparison of l0 & mesh sizes: \nl0: {length0:.8f}, fine grid: {fine_grid}\n")
print(f"length scale (m): {length0:.8f}, \nlalmbda: {lambda_:.2f}, \nmu: {mu:.2f},\nK:  {K:.2f}.")

comparison of l0 & mesh sizes: 
l0: 0.01171875, fine grid: 0.01

length scale (m): 0.01171875, 
lalmbda: 5769230769.23, 
mu: 3846153846.15,
K:  8333333333.33.


In [33]:
gmsh.initialize()
gmsh.model.add("brick") # should I even comment on that?




# bottom plane for z = 0:
p1 = gmsh.model.geo.addPoint(0, 0, 0)
p2 = gmsh.model.geo.addPoint(length, 0, 0)
p3 = gmsh.model.geo.addPoint(length, width, 0)
p4 = gmsh.model.geo.addPoint(0, width, 0)

# connecting that stuff together
l1 = gmsh.model.geo.addLine(p1, p2)
l2 = gmsh.model.geo.addLine(p2, p3)
l3 = gmsh.model.geo.addLine(p3, p4)
l4 = gmsh.model.geo.addLine(p4, p1)


# looping these
cl = gmsh.model.geo.addCurveLoop([l1, l2, l3, l4])
bottom_surface = gmsh.model.geo.addPlaneSurface([cl])

# we have a brick now, yayyy!!!
extrusion = gmsh.model.geo.extrude([(2,bottom_surface)], 0, 0, height)
gmsh.model.geo.synchronize()


volume_tag = extrusion[1][1]       # [1][] is the volume from the extrusion
top_surface_tag = extrusion[0][1]  # [0][] is the top of the extruded thing

gmsh.model.addPhysicalGroup(3, [volume_tag], tag=1)          # volume group
gmsh.model.addPhysicalGroup(2, [top_surface_tag], tag=2)     # top surface group
gmsh.model.addPhysicalGroup(2, [bottom_surface], tag=3)      # bottom surface
gmsh.model.geo.synchronize()

'''
below, I'll try to add some funny optimization: 

the brick's meshes will be denser in the middle and
thinner at sides since most of the fracture is expected to happen right below the strike. I hope. I really do. 

'''


gmsh.model.mesh.field.add('MathEval', 1)
gmsh.model.mesh.field.setString(1, "F", f"Fabs(x - {0.5 * length}) ") # our field will use the distance from the middle of the brick as the calculations' base
gmsh.model.mesh.field.add('Threshold', 2) # creates mesh size ield "Threshold" with index 2 
 
# setting the input field 
gmsh.model.mesh.field.setNumber(2, "InField", 1)

gmsh.model.mesh.field.setNumber(2, "DistMin", length*0)
gmsh.model.mesh.field.setNumber(2, "DistMax", length*0.5)
gmsh.model.mesh.field.setNumber(2, "SizeMin", fine_grid) # uses fine grid for everything that lies within DistMin
gmsh.model.mesh.field.setNumber(2, "SizeMax", coarse_grid) # uses coarse grid for everything that lies within DistMax

gmsh.model.geo.synchronize()
# setting that cursed field as the background mesh size
gmsh.model.mesh.field.setAsBackgroundMesh(2)

gmsh.model.mesh.generate(3)      



gmsh.write("brick.msh")         

gmsh.finalize()



Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 30%] Meshing curve 4 (Line)
Info    : [ 40%] Meshing curve 6 (Line)
Info    : [ 50%] Meshing curve 7 (Line)
Info    : [ 60%] Meshing curve 8 (Line)
Info    : [ 60%] Meshing curve 9 (Line)
Info    : [ 70%] Meshing curve 11 (Line)
Info    : [ 80%] Meshing curve 12 (Line)
Info    : [ 90%] Meshing curve 16 (Line)
Info    : [100%] Meshing curve 20 (Line)
Info    : Done meshing 1D (Wall 0.057925s, CPU 0.057374s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 13 (Surface, Frontal-Delaunay)
Info    : [ 40%] Meshing surface 17 (Surface, Frontal-Delaunay)
Info    : [ 60%] Meshing surface 21 (Surface, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 25 (Surface, Frontal-Delaunay)
Info    : [ 90%] Meshing surface 26 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall

In [34]:
msh, cell_tags, facet_tags, _, _, _ = gmshio.read_from_msh("brick.msh", MPI.COMM_WORLD, 0, gdim=3)
"""
just in case, that cursed thing returns the following:

Meshdata with mesh, cell tags, facet tags, edge tags, vertex tags and physical groups.

"""
# print(facet_tags, physical_groups)

Info    : Reading 'brick.msh'...
Info    : 27 entities
Info    : 162323 nodes
Info    : 991481 elements                                                                                     
Info    : Done reading 'brick.msh'                                                                               


'\njust in case, that cursed thing returns the following:\n\nMeshdata with mesh, cell tags, facet tags, edge tags, vertex tags and physical groups.\n\n'

In [35]:
# I will create needed spaces along with some initial conditions 

# V_u is vector space 
V_u = dolfinx.fem.functionspace(msh, ('Lagrange', 1, (msh.geometry.dim,)))
u_trial = TrialFunction(V_u)   
v_test = TestFunction(V_u)

# initial fields in the vector space:
u_n = fem.Function(V_u, name="displacement")
u_n_dot = fem.Function(V_u, name="velocity")
u_n_ddot = fem.Function(V_u, name="acceleration")

u_n.x.array[:] = 0.0
u_n_dot.x.array[:] = 0.0
u_n_ddot.x.array[:] = 0.0

# V_phi is a scalar space
V_phi = dolfinx.fem.functionspace(msh, ('Lagrange', 1))
psi_trial = TrialFunction(V_phi)
psi_test = TestFunction(V_phi)

# initial fields in the scalar space:
phi_n = fem.Function(V_phi, name="phase_field")
H_n = fem.Function(V_phi, name='history') 

phi_n.x.array[:] = 0.0
H_n.x.array[:] = 0.0


# creating mass matrix
dx_vertex = ufl.Measure("dx", metadata={"quadrature_rule": "vertex"})
mass_form = rho * ufl.inner(u_trial, v_test) * dx_vertex   
M = assemble_matrix(fem.form(mass_form))   
M.assemble()           

# lumping it 
diag = M.getDiagonal()
diag.assemble()
lumped_vect = diag.copy()
lumped_mass = lumped_vect.array


In [36]:
# functions needed to calculate all sorts of stuff
def epsilon(u):
    return ufl.sym(ufl.grad(u))
def effective_stress(u, lambda_ = lambda_, mu = mu):
    epsie = epsilon(u)
    return lambda_ * ufl.tr(epsie) * ufl.Identity(3) + 2 * mu * epsie

def actual_stress(u, phi):
    g = (1-phi)**2
    return (g * effective_stress(u))

def elastic_energy_density(u, mu = mu, K = K, lambda_ = lambda_): # called psi+ in the article; it's the tensile part of elastic energy
    epsie = epsilon(u)
    epsie_tr = ufl.tr(epsie)
    tr_eps_average =  0.5 * (epsie_tr + abs(epsie_tr)) # not sure whether or not that abs works 

    epsie_devatoric = epsie - epsie_tr * 1/3 * ufl.Identity(3)
    return(0.5 * K * tr_eps_average**2 + mu * ufl.inner(epsie_devatoric, epsie_devatoric))


            

In [ ]:
# dealing with the top and bottom boundaries

mesh_dim = msh.topology.dim # mesh dimenshion
facet_dim = mesh_dim - 1 # facet dimension
msh.topology.create_connectivity(facet_dim, mesh_dim)

def top_boundary(x):
    return np.isclose(x[2], 1.0, atol=1e-6)

def bottom_boundary(x):
    return np.isclose(x[2], 0.0, atol=1e-6)

top_facets = mesh.locate_entities_boundary(msh, facet_dim, top_boundary) # the meshes that satisfy the condition of a top boundary
bottom_facets = mesh.locate_entities_boundary(msh, facet_dim, bottom_boundary) # same for the bottom

top_dofs = fem.locate_dofs_topological(V_u, facet_dim, top_facets)
bottom_dofs = fem.locate_dofs_topological(V_u, facet_dim, bottom_facets)


# Dirichlet condition objects
bc_bottom = fem.dirichletbc(np.zeros(3, dtype=PETSc.ScalarType), bottom_dofs, V_u)

# now I'll set up the hit area 


V_u_z = V_u.sub(2)
# top_dofs_z = fem.locate_dofs_topological(V_u_z, facet_dim, top_facets) # the area affected by the hit

msh.topology.create_connectivity(facet_dim, 0)  # connect facets to vertices
vertex_coords = msh.geometry.x
facet_vertices = msh.topology.connectivity(facet_dim, 0)
facet_centers = np.zeros((len(top_facets), 3))

for i, f in enumerate(top_facets):
    # Get the vertices of this facet
    vertices = facet_vertices.links(f)
    # Average their coordinates
    centroid = np.mean(vertex_coords[vertices], axis=0)
    facet_centers[i] = centroid


center_x = length / 2.0
center_y = width / 2.0
center_z = height

# circular blow 
radius = 0.2
distances = np.sqrt((facet_centers[:,0] - center_x)**2 + (facet_centers[:,1] - center_y)**2)
selected_facets = top_facets[distances <= radius]


# # rectangular blow
# half_width = 0.05   # = 0.5*total width 
# half_height = 0.3  # = 0.5*total height 
# selected_facets = top_facets[(np.abs(facet_centers[:,0] - center_x) <= half_width) & (np.abs(facet_centers[:,1] - center_y) <= half_height)]
# selected_top_dofs_z = fem.locate_dofs_topological(V_u_z, facet_dim, selected_facets)

In [38]:
Path("results").mkdir(exist_ok=True)
with XDMFFile(msh.comm, "results/simulation.xdmf", "w", encoding=XDMFFile.Encoding.HDF5) as xdmf:
    xdmf.write_mesh(msh)


    # main cycle
    for time_step in range(int(time_total/dt)):
        time = dt * time_step

    # predicted value
        u_pred = fem.Function(V_u) 
        u_pred.x.array[:] = u_n.x.array[:] + dt * u_n_dot.x.array + dt**2 * 0.5 * u_n_ddot.x.array

        u_pred.x.array[bottom_dofs] = 0.0 # for the bottom to be fixed               

    # applying boundary conditions

        if time <= strike_duration:
            displacement = -velocity_0 * time
            u_pred.x.array[selected_top_dofs_z] = displacement   # top displacement

        #     bc_top = fem.dirichletbc(np.array([0.0, 0.0, displacement], dtype=PETSc.ScalarType), top_dofs, V_u)
        #     bcs = [bc_bottom, bc_top]
        # else:
        #     bcs = [bc_bottom]   # the top is not set in stone after the hit


        psi_plus = elastic_energy_density(u_pred)   
    # symbolic trial and test functions for projection
        w_trial = ufl.TrialFunction(V_phi)
        v_test_phi = ufl.TestFunction(V_phi)

    # bilinear and linear forms
        a_proj = ufl.inner(w_trial, v_test_phi) * ufl.dx
        L_proj = ufl.inner(psi_plus, v_test_phi) * ufl.dx

    # projection problem needed to find elastic energy density psi+
        problem_proj = fem.petsc.LinearProblem(a_proj, L_proj, bcs=[], petsc_options_prefix="proj")
        psi_plus_proj = problem_proj.solve()   # returns a Function

    # updating history field
        H_n.x.array[:] = np.maximum(H_n.x.array, psi_plus_proj.x.array)




    # solving phase field equation
        a_phi = (Gc / length0) * psi_trial * psi_test * ufl.dx + \
            Gc * length0 * ufl.dot(ufl.grad(psi_trial), ufl.grad(psi_test)) * ufl.dx + \
            2 * H_n * psi_trial * psi_test * ufl.dx
        L_phi = 2 * H_n * psi_test * ufl.dx

        # a_phi_comp = fem.form(a_phi)
        # L_phi_comp = fem.form(L_phi)

        problem_phi = fem.petsc.LinearProblem(a_phi, L_phi, bcs=[], petsc_options_prefix="phi")
        phi_new = problem_phi.solve() 
        # phi_new.x.array[:] = problem_phi.solve().x.array


    # calculating internal force vector
        F_int_form = ufl.inner(actual_stress(u_pred, phi_new), ufl.sym(ufl.grad(v_test))) * ufl.dx
        F_int = fem.assemble_vector(fem.form(F_int_form))

    # external force calculation
        f_ext = fem.create_vector(V_u)
        f_ext.array[:] = 0.0

        f_ext_arr = f_ext.array
        F_int_arr = F_int.array


    # acceleration calculation
        a_new_arr = (f_ext_arr - F_int_arr) / lumped_mass
        u_ddot_new = fem.Function(V_u)
        u_ddot_new.x.array[:] = a_new_arr

    # velocity calculation
        u_dot_new = fem.Function(V_u)
        u_dot_new.x.array[:] = u_n_dot.x.array + 0.5 * dt * (u_n_ddot.x.array + a_new_arr)

    # shifting fields for the next step
        u_n.x.array[:] = u_pred.x.array
        u_n_dot.x.array[:] = u_dot_new.x.array
        u_n_ddot.x.array[:] = a_new_arr
        phi_n.x.array[:] = phi_new.x.array

    # saving 
        xdmf.write_function(u_n, time)  # pass time value to write time step
        xdmf.write_function(phi_n, time)


        print(time_step)
        if time_step == 150:
            break

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150


In [39]:
# I want to create a mesh that would only include those parts of the brick that ended up in an okayish state

cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local) # creating an array of cells
cell_phi = np.zeros(len(cells))

for cell in cells:
    dofs = V_phi.dofmap.cell_dofs(cell) 
# getting out their phase field values as the mean values per cell
    cell_phi[cell] = np.mean(phi_n.x.array[dofs]) 

mask = cell_phi < 0.9
intact_cell_indices = cells[mask]
submesh = mesh.create_submesh(msh, msh.topology.dim, intact_cell_indices)[0]


In [40]:

with io.XDMFFile(MPI.COMM_WORLD, "results/healthy_brick.xdmf", "w") as xdmf:
    xdmf.write_mesh(submesh)